# Modelagem — escola–ano (Ensino Médio)

Este notebook documenta a **etapa de Machine Learning** com quatro algoritmos em papéis distintos:

| Algoritmo | Papel |
|-----------|--------|
| **HistGradientBoostingRegressor** | Regressão principal — previsão de `taxa_abandono_em`, ranking de risco e modelo final ajustado |
| **DecisionTreeRegressor** | Regras interpretáveis e thresholds para gestão escolar |
| **KNeighborsRegressor** | Escolas semelhantes / benchmarking |
| **KMeans** | Segmentação de perfis (não supervisionada nas covariáveis) |

- **Requisito 3:** ETL, EDA, pipelines (`ml/baseline_municipio.py`).
- **Requisitos 4 e 5:** validação temporal (treino `<= 2017`, teste `>= 2018`), métricas **MAE**, **RMSE**, **R²** e comparação entre regressores.
- **Requisitos 6 e 7:** ajuste de hiperparâmetros com **RandomizedSearchCV**, **TimeSeriesSplit por ano**, validação cruzada temporal e curva de aprendizado.
- **Requisitos 8 e 9:** função explícita de inferência (`edu.predict_taxa_abandono_em`) e exportação do **bundle final** em `outputs/ml/final_model_bundle.pkl`.
- **Requisito 10:** integração ao dashboard e narrativa metodológica em `ml/educational_ml.py`, `outputs/ml/` e `dashboard/app.py`.

**Granularidade:** `fato_integrado` — **uma linha por escola e ano**.  
**Alvo:** `taxa_abandono_em`. **Exclusão:** não usar `indice_risco_evasao` como preditor.


## 1. Ambiente e reprodutibilidade

Execute a partir da **raiz do repositório** (para que `data/` e `ml/` resolvam corretamente). Versões recomendadas: `requirements.txt`.

### Este notebook ainda é necessário?

**Sim.** Ele continua a ser o local onde se:
- exploram os dados (EDA);
- documentam o split temporal e as métricas;
- **geram** os ficheiros em `outputs/ml/` e `outputs/figures/` que o **dashboard** lê.

A lógica dos algoritmos está nos módulos Python (`ml/baseline_municipio.py`, `ml/educational_ml.py`); o notebook é o **roteiro reprodutível** para correr essa lógica e validar resultados.

### Onde ver as alterações (algoritmos e resultados)

| O quê | Onde |
|------|------|
| Código dos regressores (HGB, árvore, KNN) e pré-processamento | `ml/baseline_municipio.py` |
| Suite completa (comparação, KMeans, vizinhos, exportação, narrativas) | `ml/educational_ml.py` |
| Tabelas CSV + JSON para storytelling | `outputs/ml/` (`escola_ano_ml_enriquecido.csv`, `ml_storytelling.json`, …) |
| Figuras (árvore, KMeans, HGB, comparação) | `outputs/figures/` (`ml_*.png`, `model_comparison_*.png`) |
| Painel para não técnicos (após gerar os artefactos acima) | Streamlit → secção **«5. Conclusoes e Modelo Preditivo»** → bloco *Apoio inteligente* (`dashboard/app.py`) |


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "ml").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ml import baseline_municipio as bl
from ml import educational_ml as edu

## 2. Carregamento dos dados

Se `data/processed/fato_integrado.csv` não existir, o módulo **dispara o ETL automaticamente** (mesma lógica do dashboard).

In [ ]:
df = bl.load_fato_integrado()
df = df.sort_values("ano").reset_index(drop=True)
print("Dimensão:", df.shape)
print("Anos:", df["ano"].min(), "—", df["ano"].max())
df.head()

## 3. Análise exploratória inicial

Completude, distribuição do alvo e lista de colunas numéricas vs. categóricas usadas na modelagem.

In [ ]:
target = bl.TARGET
print("Taxa de valores ausentes (top 15):\n")
print(df.isna().sum().sort_values(ascending=False).head(15))

print("\n--- Estatísticas do alvo (% abandono EM) ---")
print(df[target].describe().round(3))


In [ ]:
num_cols, cat_cols = bl.infer_feature_columns(df)
print("Features numéricas:", len(num_cols))
print(num_cols)
print("\nFeatures categóricas:", cat_cols)

## 4. Tratamento e transformação (pipeline modular)

- **Numéricas:** `SimpleImputer(median)` + `StandardScaler`  
- **Categóricas** (`periodo`): `SimpleImputer(most_frequent)` + `OneHotEncoder`  
- **Regressores:** `HistGradientBoostingRegressor`, `DecisionTreeRegressor`, `KNeighborsRegressor`  
- **KMeans:** usa o mesmo pré-processamento nas covariáveis (sem o alvo nos centróides)

Implementação: `ml/baseline_municipio.py` e `ml/educational_ml.py`. Imputação ajustada **apenas no treino** (sem vazamento).


## 5. Modelo principal + validação temporal (requisito 4)

- **Treino:** anos `<= 2017` | **Teste:** anos `>= 2018`  
- Modelo único nesta secção: **HistGradientBoostingRegressor** (baseline rápido).  
- Comparação dos **três regressores** e **KMeans** na **§7**.


In [ ]:
resultado = bl.run_baseline_experiment(year_cutoff=2017)

print("Anos de treino:", resultado["train_years"])
print("Anos de teste :", resultado["test_years"])
print("n_train =", resultado["n_train"], "| n_test =", resultado["n_test"])
print()
print("--- HistGradientBoostingRegressor (modelo principal) ---")
print(resultado["metrics_hgb"])


In [ ]:
y_test = resultado["y_test"]
y_hat = resultado["y_pred_hgb"]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, y_hat, s=80, alpha=0.85, edgecolors="k")
lim = min(y_test.min(), y_hat.min()), max(y_test.max(), y_hat.max())
ax.plot(lim, lim, "r--", lw=1.5, label="y = x")
ax.set_xlabel("Abandono EM observado (%)")
ax.set_ylabel("Abandono EM previsto (%)")
ax.set_title("HistGradientBoostingRegressor — observado × previsto (teste temporal)")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Leitura dos resultados (modelo principal)

- **MAE** — erro médio em pontos percentuais (menor é melhor).
- **RMSE** — penaliza mais erros grandes.
- **R²** — variância explicada no teste (cuidado com poucas linhas no teste).
- Na **§7** comparam-se **HistGradientBoostingRegressor**, **DecisionTreeRegressor** e **KNeighborsRegressor**; o **KMeans** não entra nestas métricas (não é regressão).


## 7. Comparação dos regressores + KMeans + exportação

### 7.1 Três regressores (mesmo teste temporal)

| Modelo | Métricas no teste |
|--------|-------------------|
| HistGradientBoostingRegressor | MAE, RMSE, R² |
| DecisionTreeRegressor | MAE, RMSE, R² |
| KNeighborsRegressor | MAE, RMSE, R² |

### 7.2 KMeans (complementar)

Segmentação de escolas–ano por perfil de indicadores (cotovelo, silhueta, perfis médios por cluster).

### 7.3 Ajuste do modelo final

O **HistGradientBoostingRegressor** é ajustado com **RandomizedSearchCV** sobre o conjunto de treino, usando **TimeSeriesSplit por ano** para respeitar a ordem temporal.

### 7.4 Validação cruzada e inferência

A suite exporta:

- métricas do modelo final no **teste holdout**;
- média e desvio na **validação cruzada temporal**;
- **curva de aprendizado**;
- **bundle do modelo final** (`outputs/ml/final_model_bundle.pkl`) e função de inferência `edu.predict_taxa_abandono_em(...)`.

### 7.5 Artefatos

`outputs/figures/` (PNG) e `outputs/ml/` (CSV + JSON / PKL) — consumidos pelo dashboard (página 5).


In [ ]:
from IPython.display import display, Markdown

# --- 7.1 Comparação dos três regressores (sem Ridge / ElasticNet / Dummy) ---
cmp = bl.run_model_comparison_experiment(year_cutoff=2017)

print("Anos de treino:", cmp["train_years"], "| teste:", cmp["test_years"])
print("n_train =", cmp["n_train"], "| n_test =", cmp["n_test"])
print()
print("### Métricas no conjunto de teste temporal")
display(bl.metrics_comparison_dataframe(cmp["metrics_by_model"]))

FIG_DIR = ROOT / "outputs" / "figures"
paths_cmp = bl.plot_model_comparison_figures(
    cmp["y_test"],
    cmp["metrics_by_model"],
    cmp["predictions_by_model"],
    save_dir=FIG_DIR,
    show=True,
)
print("Figuras de comparação gravadas:", len(paths_cmp))


In [ ]:
# --- 7.2 Suite completa: KMeans, tuning, validação temporal e exportação dashboard ---
suite = edu.run_educational_ml_suite(
    year_cutoff=2017,
    save_artifacts=True,
    show_plots=True,
    permutation_repeats=8,
    tuning_iter=18,
    cv_splits=4,
)

print("### KMeans")
print("Número de clusters (k):", suite["kmeans"]["n_clusters"])
display(suite["kmeans"]["profiles"])

print()
print("### Narrativa comparativa (regressores)")
display(Markdown(suite["narrative_comparison"]))

print()
print("### Perfil dos clusters (texto)")
display(Markdown(suite["kmeans"]["profile_text"]))

print()
print("### Modelo final ajustado")
print(suite["final_model_name"])
print("Métricas no teste:", suite["final_model_test_metrics"])
print("Bundle salvo em:", suite["final_model_bundle_path"])

print()
print("Artefactos:", suite["artifact_dir"])
print("Total de figuras geradas:", len(suite["figure_paths"]))


## 8. Ajuste de hiperparâmetros do modelo final

O modelo final é o **HistGradientBoostingRegressor** ajustado por **RandomizedSearchCV** no conjunto de treino, com **TimeSeriesSplit por ano**. A tabela abaixo mostra os candidatos com menor **MAE médio de validação**.

In [ ]:
display(suite["tuning_results_table"].head(10))
print("Melhores hiperparâmetros:")
print(suite["final_model_best_params"])
print()
print("Métricas do modelo final no teste holdout:")
print(suite["final_model_test_metrics"])

## 9. Validação cruzada temporal, estabilidade e inferência

A tabela de folds abaixo mostra o comportamento do modelo final ao longo dos cortes temporais do treino. Em seguida, a função `edu.predict_taxa_abandono_em(...)` exemplifica a inferência sobre novas linhas com as mesmas features do pipeline.

In [ ]:
display(suite["final_model_cv_folds"])
print("Resumo CV:")
print(suite["final_model_cv_summary"])
print()
display(Markdown(suite["final_model_cv_diagnosis"]))
print()
print("Exemplo de inferência com 3 linhas do próprio dataset:")
base = bl.load_fato_integrado().dropna(subset=[bl.TARGET]).head(3)
pred_demo = edu.predict_taxa_abandono_em(base)
display(pred_demo[["ano", "id_linha_educacional", "pred_taxa_abandono_em"]])